In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"All imports successful (seed {SEED})")

from pathlib import Path
# Project folder, whether this runs from notebooks/ or the project root
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()


All imports successful (seed 42)


In [2]:
# Cell 2 — Build the Deck & Core Game Logic (Python translation of Java project)
# Mirrors the BlackJackToFive Java implementation from COMP 2000

SUITS = ['♠', '♥', '♦', '♣']
RANKS = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
RANK_VALUES = {
    '2':2,'3':3,'4':4,'5':5,'6':6,'7':7,'8':8,'9':9,'10':10,
    'J':10,'Q':10,'K':10,'A':11
}

def build_deck():
    return [(r, s) for s in SUITS for r in RANKS]

def hand_value(hand):
    """Mirrors BlackJackUtils ace handling — hard ace to soft ace if bust."""
    value = sum(RANK_VALUES[r] for r, s in hand)
    aces  = sum(1 for r, s in hand if r == 'A')
    while value > 21 and aces:
        value -= 10
        aces  -= 1
    return value

def is_blackjack(hand):
    return hand_value(hand) == 21 and len(hand) == 2

# Test
deck = build_deck()
random.shuffle(deck)
test_hand = [deck.pop(), deck.pop()]
print(f"Deck size:      {len(deck) + 2} cards")
print(f"Test hand:      {test_hand}")
print(f"Hand value:     {hand_value(test_hand)}")
print(f"Is blackjack:   {is_blackjack(test_hand)}")
print(f"\nCore game logic verified ✓")

Deck size:      52 cards
Test hand:      [('3', '♣'), ('9', '♠')]
Hand value:     12
Is blackjack:   False

Core game logic verified ✓


In [3]:
# Cell 3 — Simulate 10,000 BlackJack Games (Monte Carlo)
def simulate_game(n_players=3, target_wins=5, strategy='basic'):
    """
    Simulate one full BlackJack-to-Five game.
    strategy: 'basic' = hit under 17, 'aggressive' = hit under 19
    """
    deck = build_deck()
    random.shuffle(deck)
    
    wins = defaultdict(int)
    dealer_idx = 0
    rounds = 0
    hit_threshold = 17 if strategy == 'basic' else 19
    
    while max(wins.values(), default=0) < target_wins:
        if len(deck) < n_players * 4:
            deck = build_deck()
            random.shuffle(deck)
        
        hands = {i: [deck.pop(), deck.pop()] for i in range(n_players)}
        rounds += 1
        
        # Players hit or stand
        for i in range(n_players):
            if i == dealer_idx:
                continue
            while hand_value(hands[i]) < hit_threshold and len(deck) > 0:
                hands[i].append(deck.pop())
        
        # Dealer plays
        while hand_value(hands[dealer_idx]) < 17 and len(deck) > 0:
            hands[dealer_idx].append(deck.pop())
        
        dealer_val = hand_value(hands[dealer_idx])
        
        # Determine round winner
        best_val = 0
        winner   = dealer_idx
        for i in range(n_players):
            v = hand_value(hands[i])
            if v <= 21 and v > best_val:
                best_val = v
                winner   = i
        
        # Dealer wins ties (house advantage)
        if best_val == dealer_val:
            winner = dealer_idx
            
        wins[winner] += 1
        dealer_idx = (dealer_idx + 1) % n_players
    
    return wins, rounds

# Run 10,000 simulations
N_SIMS    = 10_000
N_PLAYERS = 3
results   = []

print(f"Running {N_SIMS:,} simulations with {N_PLAYERS} players...")
for _ in range(N_SIMS):
    wins, rounds = simulate_game(n_players=N_PLAYERS)
    winner = max(wins, key=wins.get)
    results.append({'winner': winner, 'rounds': rounds, 'wins': wins[winner]})

results_df = pd.DataFrame(results)

print(f"\n=== Simulation Results ===")
print(f"Avg rounds per game:     {results_df['rounds'].mean():.1f}")
print(f"Avg rounds to 5 wins:    {results_df['rounds'].mean():.1f}")
print(f"\nWin rate by player position:")
for p in range(N_PLAYERS):
    rate = (results_df['winner'] == p).mean()
    label = '(deals first)' if p == 0 else ''
    print(f"  Player {p} {label}: {rate*100:.1f}%")

Running 10,000 simulations with 3 players...



=== Simulation Results ===
Avg rounds per game:     9.7
Avg rounds to 5 wins:    9.7

Win rate by player position:
  Player 0 (deals first): 39.2%
  Player 1 : 33.9%
  Player 2 : 27.0%


In [4]:
# Cell 4 — Bust Probability (exact)
# Hitting a HARD total v busts when the next card's value exceeds 21 - v.
# An ace counts as 1 here (counting it as 11 would bust a hand that 1 saves).
# Drawing from a full 52-card deck: 4 each of 2-9 and A, 16 ten-value cards.
CARD_PROBS = {v: 4/52 for v in range(2, 10)}
CARD_PROBS[10] = 16/52
CARD_PROBS[1] = 4/52          # ace, counted as 1

bust_df = pd.DataFrame([
    {'hand_value': v,
     'bust_probability': round(sum(p for c, p in CARD_PROBS.items() if c + v > 21), 4)}
    for v in range(12, 22)
])

fig = px.bar(bust_df, x='hand_value', y='bust_probability',
             color='bust_probability',
             color_continuous_scale='Reds',
             title='Bust Probability When Hitting a Hard Total',
             labels={'hand_value': 'Current Hand Value (hard)',
                     'bust_probability': 'Probability of Bust'},
             template='plotly_dark')
fig.update_layout(width=750, height=430)
fig.show()

print("Bust probability table (exact):")
print(bust_df.to_string(index=False))


Bust probability table (exact):
 hand_value  bust_probability
         12            0.3077
         13            0.3846
         14            0.4615
         15            0.5385
         16            0.6154
         17            0.6923
         18            0.7692
         19            0.8462
         20            0.9231
         21            1.0000


In [5]:
# Cell 5 — Optimal Strategy: Dynamic Programming (Bellman equation)
# The player sees only their own hand (no dealer upcard), the dealer hits below 17,
# and ties go to the dealer. Objective: maximise the probability of winning the round.
# Infinite-deck model (each card drawn independently); Cell 6 checks it on the real deck.
from functools import lru_cache

DRAW = {v: 1/13 for v in range(2, 10)}   # 2-9
DRAW[10] = 4/13                           # 10, J, Q, K
DRAW[11] = 1/13                           # ace, counted as 11 until it would bust

def add_card(total, soft_aces, card):
    """Add a card to a hand; demote soft aces (11 -> 1) while the hand would bust."""
    total, soft_aces = total + card, soft_aces + (card == 11)
    while total > 21 and soft_aces:
        total, soft_aces = total - 10, soft_aces - 1
    return total, soft_aces

@lru_cache(maxsize=None)
def dealer_final(total, soft_aces):
    """Probability distribution of the dealer's final total (22 = bust)."""
    if total > 21:
        return {22: 1.0}
    if total >= 17:
        return {total: 1.0}
    dist = {}
    for card, p in DRAW.items():
        for final, q in dealer_final(*add_card(total, soft_aces, card)).items():
            dist[final] = dist.get(final, 0) + p * q
    return dist

DEALER = dealer_final(0, 0)

def p_win_standing(total):
    """Win if the dealer busts or ends strictly below the player's total."""
    return DEALER[22] + sum(q for final, q in DEALER.items() if final < total)

@lru_cache(maxsize=None)
def best_win_prob(total, soft_aces):
    """V(state) = max(stand, E[V(next state)]): the Bellman equation."""
    if total > 21:
        return 0.0
    hit = sum(p * best_win_prob(*add_card(total, soft_aces, c)) for c, p in DRAW.items())
    return max(p_win_standing(total), hit)

def should_hit(total, soft_aces):
    if total > 21:
        return False
    hit = sum(p * best_win_prob(*add_card(total, soft_aces, c)) for c, p in DRAW.items())
    return hit > p_win_standing(total)

@lru_cache(maxsize=None)
def threshold_win_prob(total, soft_aces, threshold):
    """Exact win probability of the simple rule 'hit while total < threshold'."""
    if total > 21:
        return 0.0
    if total >= threshold:
        return p_win_standing(total)
    return sum(p * threshold_win_prob(*add_card(total, soft_aces, c), threshold)
               for c, p in DRAW.items())

def over_starting_hands(fn):
    """Average a hand-level value over every possible two-card starting hand."""
    return sum(p * q * fn(*add_card(*add_card(0, 0, a), b))
               for a, p in DRAW.items() for b, q in DRAW.items())

optimal_win = over_starting_hands(best_win_prob)
strategy_df = pd.DataFrame([
    {'threshold': T,
     'win_rate': round(over_starting_hands(lambda t, s: threshold_win_prob(t, s, T)), 4),
     'strategy': f'Hit below {T}'}
    for T in range(12, 21)
])
best = strategy_df.loc[strategy_df['win_rate'].idxmax()]

policy_df = pd.DataFrame([
    {'total': t,
     'hard hand': 'HIT' if should_hit(t, 0) else 'stand',
     'soft hand': ('HIT' if should_hit(t, 1) else 'stand') if t >= 13 else '—'}
    for t in range(12, 22)
])

fig = go.Figure()
fig.add_trace(go.Scatter(x=strategy_df['threshold'], y=strategy_df['win_rate'],
                         mode='lines+markers', line=dict(color='cyan', width=2.5),
                         marker=dict(size=8), name='Fixed threshold (exact)'))
fig.add_hline(y=optimal_win, line_dash='dash', line_color='lime',
              annotation_text=f'DP optimal policy: {optimal_win:.1%}')
fig.update_layout(title='Win Probability by Strategy — exact dynamic programming',
                  xaxis_title='Hit threshold (stand at or above this value)',
                  yaxis_title='Probability of winning the round',
                  template='plotly_dark', width=800, height=450)
fig.show()

print(f"Dealer bust probability:            {DEALER[22]:.1%}")
print(f"Best fixed rule:                    {best['strategy']} ({best['win_rate']:.1%})")
print(f"DP optimal policy (hard + soft):    {optimal_win:.1%}")
print("\nOptimal policy by hand total:")
print(policy_df.to_string(index=False))


Dealer bust probability:            28.2%
Best fixed rule:                    Hit below 14 (41.8%)
DP optimal policy (hard + soft):    42.5%

Optimal policy by hand total:
 total hard hand soft hand
    12       HIT         —
    13       HIT       HIT
    14     stand       HIT
    15     stand       HIT
    16     stand       HIT
    17     stand       HIT
    18     stand       HIT
    19     stand     stand
    20     stand     stand
    21     stand     stand


In [6]:
# Cell 6 — Validate on the real deck & compare strategies
# Plays the game's actual rules (fresh 52-card deck each hand). Common random numbers:
# every strategy plays the SAME shuffled deck, so their differences are measured precisely.
import os, math

def hand_soft(hand):
    """True if the hand holds an ace still counted as 11."""
    total = sum(RANK_VALUES[r] for r, s in hand)
    aces = sum(1 for r, s in hand if r == 'A')
    while total > 21 and aces:
        total, aces = total - 10, aces - 1
    return aces > 0

def play_round(deck, policy):
    d = list(deck)
    hand = [d.pop(), d.pop()]
    while policy(hand_value(hand), hand_soft(hand)):
        hand.append(d.pop())
    dealer = [d.pop(), d.pop()]
    while hand_value(dealer) < 17:
        dealer.append(d.pop())
    p, q = hand_value(hand), hand_value(dealer)
    return int(p <= 21 and (q > 21 or p > q))

policies = {
    'Conservative (hit<17)': lambda t, s: t < 17,
    'Basic (hit<18)':        lambda t, s: t < 18,
    'Aggressive (hit<19)':   lambda t, s: t < 19,
    f'Best fixed (hit<{int(best["threshold"])})': lambda t, s, T=int(best['threshold']): t < T,
    'Optimal (DP policy)':   lambda t, s: should_hit(t, int(s)),
}

N_HANDS = 100_000
rng = random.Random(SEED)
outcomes = {name: np.zeros(N_HANDS, dtype=np.int8) for name in policies}
for i in range(N_HANDS):
    deck = build_deck()
    rng.shuffle(deck)
    for name, policy in policies.items():
        outcomes[name][i] = play_round(deck, policy)

exact = {'Conservative (hit<17)': 17, 'Basic (hit<18)': 18, 'Aggressive (hit<19)': 19}
rows = []
for name, wins in outcomes.items():
    p = wins.mean()
    if name == 'Optimal (DP policy)':
        model = optimal_win
    elif name.startswith('Best fixed'):
        model = best['win_rate']
    else:
        model = strategy_df.set_index('threshold').loc[exact[name], 'win_rate']
    rows.append({'strategy': name, 'win_rate': round(p, 4),
                 'ci95': round(1.96 * math.sqrt(p * (1 - p) / N_HANDS), 4),
                 'dp_model': round(float(model), 4)})
compare_df = pd.DataFrame(rows)

d = outcomes['Optimal (DP policy)'].astype(float) - outcomes['Conservative (hit<17)']
gain, gain_ci = d.mean(), 1.96 * d.std(ddof=1) / math.sqrt(N_HANDS)

print(f"=== Strategy Comparison — {N_HANDS:,} hands, real single deck ===")
print(compare_df.to_string(index=False))
print(f"\nOptimal minus Conservative: {gain:+.2%} ± {gain_ci:.2%} (95% CI, paired)")

fig2 = px.bar(compare_df, x='strategy', y='win_rate', error_y='ci95',
              color='win_rate', color_continuous_scale='Viridis',
              title='Win Rate by Strategy (simulated, 95% CI)',
              template='plotly_dark')
fig2.update_layout(width=750, height=430)
fig2.show()

# Export
output_dir = str(PROJECT_DIR / 'outputs')
os.makedirs(output_dir, exist_ok=True)
results_df.to_csv(f'{output_dir}/simulation_results.csv', index=False)
bust_df.to_csv(f'{output_dir}/bust_probabilities.csv', index=False)
strategy_df.to_csv(f'{output_dir}/strategy_optimization.csv', index=False)
policy_df.to_csv(f'{output_dir}/optimal_policy.csv', index=False)
compare_df.to_csv(f'{output_dir}/strategy_comparison.csv', index=False)
print(f"\nAll exports saved to outputs/")


=== Strategy Comparison — 100,000 hands, real single deck ===
             strategy  win_rate   ci95  dp_model
Conservative (hit<17)    0.4078 0.0030    0.4077
       Basic (hit<18)    0.3976 0.0030    0.3984
  Aggressive (hit<19)    0.3608 0.0030    0.3617
  Best fixed (hit<14)    0.4190 0.0031    0.4180
  Optimal (DP policy)    0.4259 0.0031    0.4250

Optimal minus Conservative: +1.80% ± 0.23% (95% CI, paired)



All exports saved to outputs/


In [7]:
# Cell 7 — Summary (computed from the results above)
first_dealer = (results_df['winner'] == 0).mean()
print(f"""
BlackJack to Five — Project Summary
  Implementation:   Python translation of the Java COMP 2000 game (seed {SEED})
  Simulations:      {N_SIMS:,} full games, 3 players, rotating dealer

  Player who deals first wins {first_dealer:.1%} of games (ties go to the dealer)
  Average game length: {results_df['rounds'].mean():.1f} rounds
  Bust chance when hitting: {bust_df.set_index('hand_value').loc[12,'bust_probability']:.1%} at hard 12,
                            {bust_df.set_index('hand_value').loc[20,'bust_probability']:.1%} at hard 20
  Best fixed rule: {best['strategy']} ({best['win_rate']:.1%})
  DP optimal policy: {optimal_win:.1%}, {gain:+.1%} vs 'hit below 17' (95% CI ±{gain_ci:.1%})
""")



BlackJack to Five — Project Summary
  Implementation:   Python translation of the Java COMP 2000 game (seed 42)
  Simulations:      10,000 full games, 3 players, rotating dealer

  Player who deals first wins 39.2% of games (ties go to the dealer)
  Average game length: 9.7 rounds
  Bust chance when hitting: 30.8% at hard 12,
                            92.3% at hard 20
  Best fixed rule: Hit below 14 (41.8%)
  DP optimal policy: 42.5%, +1.8% vs 'hit below 17' (95% CI ±0.2%)

